In [1]:
%pip install -q "lerobot[smolvla,dataset]==0.6.1" "transformers==5.5.0" "accelerate>=1.12.0" "safetensors>=0.5" "av" "pyarrow" "pandas"

  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.14.1 requires packaging>=26.0.0, but you have packaging 25.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


In [2]:
# CELL 1 — LFM2.5-VL ARCHITECTURE PREFLIGHT

import torch
import transformers
import numpy as np
import hashlib
import gc

from transformers import AutoConfig
from huggingface_hub import HfApi

LFM_ID = "LiquidAI/LFM2.5-VL-450M"
NATIVE_ID = "lerobot/smolvla_base"

DEPTHS = (8, 10, 14)
EXPERT_LAYERS = 16

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16 if torch.cuda.is_available()
    else torch.float32
)

print("=" * 65)
print("LFM2.5-VL-450M ARCHITECTURE PRECHECK")
print("=" * 65)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", DEVICE)
print("Dtype:", DTYPE)

assert DEVICE.type == "cuda", "Enable Kaggle GPU"

lfm_cfg = AutoConfig.from_pretrained(LFM_ID)
text_cfg = lfm_cfg.text_config
vision_cfg = lfm_cfg.vision_config

print("\n=== VLM BACKBONE ===")
print("Architecture:", lfm_cfg.architectures)
print("Text hidden:", text_cfg.hidden_size)
print("Text layers:", text_cfg.num_hidden_layers)
print("Attention heads:", text_cfg.num_attention_heads)
print("KV heads:", text_cfg.num_key_value_heads)

print("\n=== VISION ENCODER ===")
print("Vision hidden:", vision_cfg.hidden_size)
print("Vision layers:", vision_cfg.num_hidden_layers)

print("\n=== LAYER TYPES ===")
for i, kind in enumerate(text_cfg.layer_types):
    print(f"Layer {i:02d}: {kind}")

assert text_cfg.hidden_size == 1024
assert text_cfg.num_hidden_layers == 16
assert len(text_cfg.layer_types) == 16
assert all(d <= 16 for d in DEPTHS)

print("\n=== TRUNCATION PLAN ===")

for d in DEPTHS:
    types = text_cfg.layer_types[:d]

    print(
        f"L{d}: {len(types)} layers | "
        f"Attention={types.count('full_attention')} | "
        f"Conv={types.count('conv')}"
    )

print("\n[PASS] LFM architecture precheck")

LFM2.5-VL-450M ARCHITECTURE PRECHECK
PyTorch: 2.11.0+cu128
Transformers: 5.5.0
Device: cuda:0
Dtype: torch.bfloat16


config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]


=== VLM BACKBONE ===
Architecture: ['Lfm2VlForConditionalGeneration']
Text hidden: 1024
Text layers: 16
Attention heads: 16
KV heads: 8

=== VISION ENCODER ===
Vision hidden: 768
Vision layers: 12

=== LAYER TYPES ===
Layer 00: conv
Layer 01: conv
Layer 02: full_attention
Layer 03: conv
Layer 04: conv
Layer 05: full_attention
Layer 06: conv
Layer 07: conv
Layer 08: full_attention
Layer 09: conv
Layer 10: full_attention
Layer 11: conv
Layer 12: full_attention
Layer 13: conv
Layer 14: full_attention
Layer 15: conv

=== TRUNCATION PLAN ===
L8: 8 layers | Attention=2 | Conv=6
L10: 10 layers | Attention=3 | Conv=7
L14: 14 layers | Attention=5 | Conv=9

[PASS] LFM architecture precheck


In [3]:
# CELL 2 — NATIVE SMOLVLA EXPERT AUDIT

from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy
)

print("=" * 65)
print("NATIVE SMOLVLA EXPERT AUDIT")
print("=" * 65)

if "policy" not in globals():
    policy = SmolVLAPolicy.from_pretrained(
        NATIVE_ID
    ).to(DEVICE)

policy.eval()

m = policy.model
ve = m.vlm_with_expert
expert = ve.lm_expert

print("Expert class:", type(expert).__name__)
print("Attention mode:", ve.attention_mode)
print("Expert layers:", len(expert.layers))
print("Expert hidden:", ve.expert_hidden_size)
print("Native VLM hidden:", ve.vlm.config.text_config.hidden_size)
print("Native VLM layers:", ve.num_vlm_layers)
print("Expert KV heads:", ve.vlm.config.text_config.num_key_value_heads)
print("Head dimension:", ve.vlm.config.text_config.head_dim)

print("\n=== ACTION INTERFACE ===")
print("Chunk:", m.config.chunk_size)
print("Max action dimension:", m.config.max_action_dim)
print("Max state dimension:", m.config.max_state_dim)

expert_params = sum(
    p.numel() for p in expert.parameters()
)

print("\nExpert parameters:", f"{expert_params:,}")

print("\n=== SELECTED EXPERT LAYERS ===")

for idx in (0, 7, 15):
    attn = expert.layers[idx].self_attn

    print(
        f"Layer {idx}:",
        "Q", tuple(attn.q_proj.weight.shape),
        "K", tuple(attn.k_proj.weight.shape),
        "V", tuple(attn.v_proj.weight.shape),
        "O", tuple(attn.o_proj.weight.shape),
    )

assert ve.attention_mode == "cross_attn"
assert len(expert.layers) == 16
assert ve.expert_hidden_size == 720
assert m.config.chunk_size == 50
assert m.config.max_action_dim == 32

print("\n[PASS] Original SmolVLA Expert preserved")

NATIVE SMOLVLA EXPERT AUDIT


config.json:   0%|          | 0.00/2.30k [00:00<?, ?B/s]

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

tokenizer.json:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

Reducing the number of VLM layers to 16 ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  907MB            

model.safetensors: downloading bytes:           |  0.00B            

Expert class: LlamaModel
Attention mode: cross_attn
Expert layers: 16
Expert hidden: 720
Native VLM hidden: 960
Native VLM layers: 16
Expert KV heads: 5
Head dimension: 64

=== ACTION INTERFACE ===
Chunk: 50
Max action dimension: 32
Max state dimension: 32

Expert parameters: 98,245,840

=== SELECTED EXPERT LAYERS ===
Layer 0: Q (960, 720) K (320, 720) V (320, 720) O (720, 960)
Layer 7: Q (960, 720) K (320, 320) V (320, 320) O (720, 960)
Layer 15: Q (960, 720) K (320, 320) V (320, 320) O (720, 960)

[PASS] Original SmolVLA Expert preserved


In [4]:
# CELL 3 — LOAD REAL LFM2.5-VL-450M

import torch.nn as nn

from transformers import (
    AutoProcessor,
    Lfm2VlForConditionalGeneration,
)

print("=" * 65)
print("LOADING LFM2.5-VL-450M")
print("=" * 65)

lfm_processor = AutoProcessor.from_pretrained(
    LFM_ID
)

lfm = Lfm2VlForConditionalGeneration.from_pretrained(
    LFM_ID,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)

lfm.eval()

# Native LFM2.5 multimodal backbone
lfm_base = lfm.model
lfm_lm = lfm_base.language_model

assert len(lfm_lm.layers) == 16
assert lfm_lm.config.hidden_size == 1024

print("\n=== LOADED COMPONENTS ===")
print("Model:", type(lfm).__name__)
print("Vision:", type(lfm_base.vision_tower).__name__)
print("Projector:", type(lfm_base.multi_modal_projector).__name__)
print("Language:", type(lfm_lm).__name__)

print("\n=== LANGUAGE LAYERS ===")

for idx, layer in enumerate(lfm_lm.layers):
    print(
        f"Layer {idx:02d}:",
        "Attention" if layer.is_attention_layer
        else "Convolution"
    )

total_params = sum(
    p.numel() for p in lfm.parameters()
)

print("\nTotal LFM parameters:", f"{total_params:,}")

assert total_params > 0

print("\n[PASS] Pretrained LFM2.5 loaded")

LOADING LFM2.5-VL-450M


processor_config.json:   0%|          | 0.00/828 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.84k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  897MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/349 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]


=== LOADED COMPONENTS ===
Model: Lfm2VlForConditionalGeneration
Vision: Siglip2VisionModel
Projector: Lfm2VlMultiModalProjector
Language: Lfm2Model

=== LANGUAGE LAYERS ===
Layer 00: Convolution
Layer 01: Convolution
Layer 02: Attention
Layer 03: Convolution
Layer 04: Convolution
Layer 05: Attention
Layer 06: Convolution
Layer 07: Convolution
Layer 08: Attention
Layer 09: Convolution
Layer 10: Attention
Layer 11: Convolution
Layer 12: Attention
Layer 13: Convolution
Layer 14: Attention
Layer 15: Convolution

Total LFM parameters: 448,718,848

[PASS] Pretrained LFM2.5 loaded


In [5]:
# CELL 4 — THREE CAMERA + LANGUAGE FEATURE TEST

from PIL import Image

print("=" * 65)
print("LFM MULTIMODAL FEATURE TEST")
print("=" * 65)

CAM_NAMES = [
    "cam_head",
    "cam_side",
    "cam_wrist",
]

INSTRUCTION = (
    "pick up the orange cube and place it into the bowl"
)

if (
    "images" in globals()
    and isinstance(images, (list, tuple))
    and len(images) == 3
    and all(isinstance(im, Image.Image) for im in images)
):
    lfm_images = [im.convert("RGB") for im in images]
    print("Using existing real camera images")
else:
    lfm_images = [
        Image.new("RGB", (256, 256), (180, 80, 40)),
        Image.new("RGB", (256, 256), (80, 180, 40)),
        Image.new("RGB", (256, 256), (40, 80, 180)),
    ]
    print("Using interface-test images only")

messages = [{
    "role": "user",
    "content": [
        {"type": "text", "text": "Head camera:"},
        {"type": "image", "image": lfm_images[0]},
        {"type": "text", "text": "Side camera:"},
        {"type": "image", "image": lfm_images[1]},
        {"type": "text", "text": "Wrist camera:"},
        {"type": "image", "image": lfm_images[2]},
        {"type": "text", "text": INSTRUCTION},
    ],
}]

inputs = lfm_processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {
    k: (
        v.to(DEVICE, dtype=DTYPE)
        if k == "pixel_values"
        else v.to(DEVICE)
        if isinstance(v, torch.Tensor)
        else v
    )
    for k, v in inputs.items()
}

print("\n=== PROCESSOR OUTPUTS ===")
for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)

assert "input_ids" in inputs
assert "pixel_values" in inputs

# Capture all 16 hidden states using actual LFM blocks.
lfm_captured = {}
hooks = []

def make_lfm_hook(index):
    def hook(module, args, output):
        hidden = (
            output[0]
            if isinstance(output, (tuple, list))
            else output
        )
        lfm_captured[index] = hidden.detach()
    return hook

for i in range(16):
    hooks.append(
        lfm_lm.layers[i].register_forward_hook(
            make_lfm_hook(i)
        )
    )

try:
    with torch.inference_mode():
        out = lfm(
            **inputs,
            use_cache=False,
            logits_to_keep=1,
            return_dict=True,
        )
finally:
    for h in hooks:
        h.remove()

assert len(lfm_captured) == 16

lfm_features = [
    lfm_captured[i]
    for i in range(16)
]

print("\n=== LFM OUTPUT FEATURES ===")

for i in (0, 7, 9, 13, 15):
    x = lfm_features[i]
    print(f"Layer {i}:", tuple(x.shape))

assert all(
    x.ndim == 3
    and x.shape[0] == 1
    and x.shape[-1] == 1024
    and torch.isfinite(x).all()
    for x in lfm_features
)

print("\n[PASS] Three-camera processor")
print("[PASS] Real pretrained LFM forward")
print("[PASS] 16 intermediate hidden states")
print("[PASS] Hidden dimension 1024")

LFM MULTIMODAL FEATURE TEST
Using interface-test images only



=== PROCESSOR OUTPUTS ===
pixel_values (3, 1024, 768) torch.bfloat16
pixel_attention_mask (3, 1024) torch.int32
spatial_shapes (3, 2) torch.int64
input_ids (1, 229) torch.int64
attention_mask (1, 229) torch.int64



=== LFM OUTPUT FEATURES ===
Layer 0: (1, 229, 1024)
Layer 7: (1, 229, 1024)
Layer 9: (1, 229, 1024)
Layer 13: (1, 229, 1024)
Layer 15: (1, 229, 1024)

[PASS] Three-camera processor
[PASS] Real pretrained LFM forward
[PASS] 16 intermediate hidden states
[PASS] Hidden dimension 1024


In [6]:
# CELL 5 — LFM -> NATIVE EXPERT COMPATIBILITY

import torch.nn as nn
from transformers import DynamicCache

print("=" * 65)
print("LFM -> NATIVE SMOLVLA COMPATIBILITY")
print("=" * 65)

LFM_HIDDEN = 1024
EXPERT_HIDDEN = 720

KV_HEADS = 5
HEAD_DIM = 64
KV_WIDTH = KV_HEADS * HEAD_DIM

N_EXPERT_LAYERS = 16

assert KV_WIDTH == 320
assert len(expert.layers) == 16
assert len(lfm_features) == 16

# Mapping 16 native expert layers to truncated LFM layers.
def layer_mapping(depth):
    return [
        min(depth - 1, (i * depth) // 16)
        for i in range(16)
    ]

for depth in DEPTHS:
    mapping = layer_mapping(depth)

    assert len(mapping) == 16
    assert min(mapping) >= 0
    assert max(mapping) == depth - 1

    print(f"\nL{depth} -> 16 Expert layers")
    print(mapping)


class LFMKVAdapter(nn.Module):
    def __init__(self):
        super().__init__()

        self.projections = nn.ModuleList([
            nn.ModuleDict({
                "key": nn.Linear(1024, 320),
                "value": nn.Linear(1024, 320),
            })
            for _ in range(16)
        ])

    def forward(self, features, depth):
        cache = DynamicCache()
        mapping = layer_mapping(depth)

        for expert_idx, source_idx in enumerate(mapping):
            x = features[source_idx]

            proj = self.projections[expert_idx]
            x = x.to(proj["key"].weight.dtype)

            k = proj["key"](x)
            v = proj["value"](x)

            B, P, _ = k.shape

            k = (
                k.reshape(B, P, KV_HEADS, HEAD_DIM)
                .transpose(1, 2)
                .contiguous()
            )

            v = (
                v.reshape(B, P, KV_HEADS, HEAD_DIM)
                .transpose(1, 2)
                .contiguous()
            )

            cache.update(k, v, expert_idx)

        return cache


lfm_adapter = LFMKVAdapter().to(
    device=DEVICE,
    dtype=DTYPE,
)

print("\n=== ADAPTER ===")

print(
    "Parameters:",
    f"{sum(p.numel() for p in lfm_adapter.parameters()):,}"
)

# Precheck only: features were captured from a full
# 16-layer run, NOT from genuinely truncated models.

for depth in DEPTHS:
    with torch.no_grad():
        cache = lfm_adapter(
            lfm_features,
            depth,
        )

    assert len(cache.layers) == 16

    for layer in cache.layers:
        assert layer.keys.shape[1] == 5
        assert layer.keys.shape[-1] == 64
        assert layer.values.shape == layer.keys.shape

    print(
        f"[PASS] L{depth}:",
        "16 KV sets,",
        tuple(cache.layers[0].keys.shape)
    )

print("\n[PASS] Dimension compatibility")
print("[PASS] Native expert unchanged")
print("[INFO] No backbone/expert integration yet")

LFM -> NATIVE SMOLVLA COMPATIBILITY

L8 -> 16 Expert layers
[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7]

L10 -> 16 Expert layers
[0, 0, 1, 1, 2, 3, 3, 4, 5, 5, 6, 6, 7, 8, 8, 9]

L14 -> 16 Expert layers
[0, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 11, 12, 13]

=== ADAPTER ===
Parameters: 10,496,000
[PASS] L8: 16 KV sets, (1, 5, 229, 64)
[PASS] L10: 16 KV sets, (1, 5, 229, 64)
[PASS] L14: 16 KV sets, (1, 5, 229, 64)

[PASS] Dimension compatibility
[PASS] Native expert unchanged
[INFO] No backbone/expert integration yet


In [7]:
# ============================================================
# CELL 6 — REAL TRUNCATED LFM -> NATIVE SMOLVLA EXPERT
# ============================================================
# L8 / L10 / L14 actually execute 8 / 10 / 14 layers.
# Uses multimodal inputs prepared in Cell 4.
#
# Tests:
#   - True backbone truncation
#   - LFM hidden features
#   - Robot state conditioning
#   - 16-layer KV Adapter
#   - Native SmolVLA denoise_step
#   - Gradient flow
#
# No optimizer updates.
# No checkpoint changes.
# Images are still interface-test images if Cell 4
# reported "Using interface-test images only".
# ============================================================

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import DynamicCache

assert "lfm" in globals()
assert "lfm_lm" in globals()
assert "lfm_adapter" in globals()
assert "inputs" in globals()
assert "policy" in globals()

m = policy.model
ve = m.vlm_with_expert
expert = ve.lm_expert

DEPTHS = (8, 10, 14)

DEVICE = next(expert.parameters()).device
ADAPTER_DTYPE = next(lfm_adapter.parameters()).dtype

assert len(lfm_lm.layers) == 16
assert len(expert.layers) == 16
assert ve.attention_mode == "cross_attn"

print("=" * 70)
print("LFM TRUE TRUNCATION -> NATIVE SMOLVLA EXPERT")
print("=" * 70)

# ============================================================
# 1. SAVE ORIGINAL BACKBONE CONFIGURATION
# ============================================================

original_layers = lfm_lm.layers

original_lm_depth = lfm_lm.config.num_hidden_layers
original_text_depth = lfm.config.text_config.num_hidden_layers

original_cache_setting = m.config.use_cache

original_native_flags = [
    p.requires_grad
    for p in m.parameters()
]

original_lfm_flags = [
    p.requires_grad
    for p in lfm.parameters()
]

# ============================================================
# 2. STATE COMPATIBILITY BRIDGE
# ============================================================
# Native SmolVLA state projection:
#   32 -> 960
#
# Compatibility bridge:
#   960 -> 1024
#
# Native state projection remains unchanged.
# ============================================================

lfm_state_bridge = nn.Linear(
    m.state_proj.out_features,
    1024,
).to(
    device=DEVICE,
    dtype=ADAPTER_DTYPE,
)

print("\nState projection:", m.state_proj)
print("State bridge:", lfm_state_bridge)

# Placeholder state for interface test only.
# Real SO-ARM101 state will be loaded from dataset later.

state6 = torch.zeros(
    1, 6,
    device=DEVICE,
    dtype=torch.float32,
)

# ============================================================
# 3. TRUE TRUNCATED FEATURE EXTRACTION
# ============================================================

def extract_truncated_lfm(depth):

    assert depth in DEPTHS
    assert len(lfm_lm.layers) == depth

    captured = {}
    hooks = []

    def make_hook(index):

        def hook(module, args, output):

            hidden = (
                output[0]
                if isinstance(output, (tuple, list))
                else output
            )

            captured[index] = hidden

        return hook

    for i in range(depth):
        hooks.append(
            lfm_lm.layers[i].register_forward_hook(
                make_hook(i)
            )
        )

    try:
        # no_grad rather than inference_mode:
        # outputs can safely enter a trainable adapter.
        with torch.no_grad():

            _ = lfm(
                **inputs,
                use_cache=False,
                logits_to_keep=1,
                return_dict=True,
            )

        assert len(captured) == depth, (
            f"Expected {depth} layers, "
            f"captured {len(captured)}"
        )

        features = [
            captured[i].detach().clone()
            for i in range(depth)
        ]

    finally:
        for h in hooks:
            h.remove()

    for i, x in enumerate(features):

        assert x.ndim == 3
        assert x.shape[0] == 1
        assert x.shape[-1] == 1024
        assert torch.isfinite(x).all(), (
            f"Invalid LFM layer {i}"
        )

    return features


# ============================================================
# 4. BUILD STATE-AWARE KV CACHE
# ============================================================

def build_native_cache(features, depth, robot_state):

    assert len(features) == depth

    # Pad 6D state to native SmolVLA 32D state format.
    state32 = F.pad(
        robot_state.float(),
        (0, 26),
    )

    # Original SmolVLA state projection.
    state32 = state32.to(
        device=m.state_proj.weight.device,
        dtype=m.state_proj.weight.dtype,
    )

    native_state = m.state_proj(state32)

    # Compatibility bridge only.
    state_token = lfm_state_bridge(
        native_state.to(
            device=lfm_state_bridge.weight.device,
            dtype=lfm_state_bridge.weight.dtype,
        )
    ).unsqueeze(1)

    # Append robot state to every selected LFM layer output.
    state_aware_features = []

    for x in features:

        x = x.to(
            device=state_token.device,
            dtype=state_token.dtype,
        )

        combined = torch.cat(
            [x, state_token],
            dim=1,
        )

        state_aware_features.append(combined)

    # Native-compatible 16-layer cache.
    cache = lfm_adapter(
        state_aware_features,
        depth,
    )

    assert len(cache.layers) == 16

    expected_prefix_len = (
        features[0].shape[1] + 1
    )

    for i, layer in enumerate(cache.layers):

        expected = (
            1,
            5,
            expected_prefix_len,
            64,
        )

        assert tuple(layer.keys.shape) == expected, (
            f"Layer {i} K shape mismatch"
        )

        assert tuple(layer.values.shape) == expected, (
            f"Layer {i} V shape mismatch"
        )

    return cache


# ============================================================
# 5. NATIVE SMOLVLA FORWARD
# ============================================================

def native_lfm_forward(cache, noisy_actions, timestep):

    B, T, D = noisy_actions.shape

    assert T == 50
    assert D == 32

    prefix_len = cache.layers[0].keys.shape[2]

    prefix_pad_masks = torch.ones(
        B,
        prefix_len,
        dtype=torch.bool,
        device=noisy_actions.device,
    )

    # Native Action Expert + native denoising method.
    velocity = m.denoise_step(
        prefix_pad_masks=prefix_pad_masks,
        past_key_values=cache,
        x_t=noisy_actions.to(
            device=m.action_in_proj.weight.device,
            dtype=m.action_in_proj.weight.dtype,
        ),
        timestep=timestep.to(
            device=noisy_actions.device,
            dtype=torch.float32,
        ),
    )

    return velocity


# ============================================================
# 6. FORWARD / BACKWARD EACH DEPTH
# ============================================================

results = {}

try:

    # Freeze model weights for this interface test.
    # Autograd still propagates through them to adapter.
    for p in m.parameters():
        p.requires_grad_(False)

    for p in lfm.parameters():
        p.requires_grad_(False)

    m.eval()
    lfm.eval()

    m.config.use_cache = True

    for depth in DEPTHS:

        print("\n" + "-" * 70)
        print(f"TESTING L{depth}")
        print("-" * 70)

        # ----------------------------------------------------
        # ACTUALLY TRUNCATE THE BACKBONE
        # ----------------------------------------------------

        lfm_lm.layers = nn.ModuleList(
            list(original_layers)[:depth]
        )

        lfm_lm.config.num_hidden_layers = depth
        lfm.config.text_config.num_hidden_layers = depth

        assert len(lfm_lm.layers) == depth

        print("Executed layers:", len(lfm_lm.layers))

        # ----------------------------------------------------
        # REAL PRETRAINED LFM FORWARD
        # ----------------------------------------------------

        features = extract_truncated_lfm(depth)

        print(
            "Backbone feature:",
            tuple(features[-1].shape),
        )

        assert len(features) == depth

        # ----------------------------------------------------
        # ADAPTER FORWARD
        # ----------------------------------------------------

        lfm_adapter.zero_grad(set_to_none=True)
        lfm_state_bridge.zero_grad(set_to_none=True)

        cache = build_native_cache(
            features,
            depth,
            state6,
        )

        print(
            "Native KV shape:",
            tuple(cache.layers[0].keys.shape),
        )

        # ----------------------------------------------------
        # NATIVE EXPERT FORWARD
        # ----------------------------------------------------

        noisy_actions = torch.randn(
            1,
            50,
            32,
            device=DEVICE,
            dtype=m.action_in_proj.weight.dtype,
        )

        timestep = torch.tensor(
            [0.5],
            device=DEVICE,
            dtype=torch.float32,
        )

        velocity = native_lfm_forward(
            cache,
            noisy_actions,
            timestep,
        )

        assert velocity.shape == (1, 50, 32)
        assert torch.isfinite(velocity).all()

        print(
            "Native velocity:",
            tuple(velocity.shape),
        )

        # ----------------------------------------------------
        # BACKWARD TEST
        # ----------------------------------------------------

        target = torch.randn_like(
            velocity,
            dtype=torch.float32,
        )

        loss = F.mse_loss(
            velocity.float(),
            target,
        )

        assert torch.isfinite(loss)

        loss.backward()

        adapter_grads = [
            p.grad
            for p in lfm_adapter.parameters()
            if p.grad is not None
        ]

        state_grads = [
            p.grad
            for p in lfm_state_bridge.parameters()
            if p.grad is not None
        ]

        assert adapter_grads, "No KV adapter gradients"
        assert state_grads, "No state bridge gradients"

        assert all(
            torch.isfinite(g).all()
            for g in adapter_grads
        )

        assert all(
            torch.isfinite(g).all()
            for g in state_grads
        )

        adapter_grad_norm = sum(
            g.float().norm().item()
            for g in adapter_grads
        )

        assert adapter_grad_norm > 0

        results[f"L{depth}"] = {
            "executed_layers": depth,
            "feature_tokens": features[-1].shape[1],
            "kv_layers": len(cache.layers),
            "velocity_shape": tuple(velocity.shape),
            "loss": float(loss.item()),
            "adapter_grad_norm": adapter_grad_norm,
        }

        print(
            f"[PASS] L{depth} "
            f"| Loss={loss.item():.6f} "
            f"| Grad={adapter_grad_norm:.6f}"
        )

        del (
            features,
            cache,
            velocity,
            loss,
            noisy_actions,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

finally:

    # --------------------------------------------------------
    # RESTORE ORIGINAL LFM BACKBONE
    # --------------------------------------------------------

    lfm_lm.layers = original_layers

    lfm_lm.config.num_hidden_layers = (
        original_lm_depth
    )

    lfm.config.text_config.num_hidden_layers = (
        original_text_depth
    )

    m.config.use_cache = original_cache_setting

    for p, flag in zip(
        m.parameters(),
        original_native_flags,
    ):
        p.requires_grad_(flag)

    for p, flag in zip(
        lfm.parameters(),
        original_lfm_flags,
    ):
        p.requires_grad_(flag)

# ============================================================
# 7. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("CELL 6 — INTEGRATION RESULTS")
print("=" * 70)

for name, value in results.items():

    print(
        name,
        "| Executed:",
        value["executed_layers"],
        "| Tokens:",
        value["feature_tokens"],
        "| Expert KV:",
        value["kv_layers"],
        "| Velocity:",
        value["velocity_shape"],
        "| Loss:",
        round(value["loss"], 6),
    )

assert set(results) == {"L8", "L10", "L14"}

assert len(lfm_lm.layers) == 16
assert len(expert.layers) == 16

print("\n[PASS] L8 executed 8 layers")
print("[PASS] L10 executed 10 layers")
print("[PASS] L14 executed 14 layers")
print("[PASS] Native Expert 16 layers unchanged")
print("[PASS] 16 KV cache layers")
print("[PASS] Native velocity shape 50 x 32")
print("[PASS] Adapter backward")
print("[PASS] No model weights updated")
print("[PASS] Original full LFM model restored")

print("\nCELL 6 COMPLETE")

LFM TRUE TRUNCATION -> NATIVE SMOLVLA EXPERT

State projection: Linear(in_features=32, out_features=960, bias=True)
State bridge: Linear(in_features=960, out_features=1024, bias=True)

----------------------------------------------------------------------
TESTING L8
----------------------------------------------------------------------
Executed layers: 8


Backbone feature: (1, 229, 1024)
Native KV shape: (1, 5, 230, 64)


Native velocity: (1, 50, 32)
[PASS] L8 | Loss=1.830594 | Grad=130.893499



----------------------------------------------------------------------
TESTING L10
----------------------------------------------------------------------
Executed layers: 10


Backbone feature: (1, 229, 1024)
Native KV shape: (1, 5, 230, 64)
Native velocity: (1, 50, 32)
[PASS] L10 | Loss=1.822181 | Grad=129.419545



----------------------------------------------------------------------
TESTING L14
----------------------------------------------------------------------
Executed layers: 14


Backbone feature: (1, 229, 1024)
Native KV shape: (1, 5, 230, 64)
Native velocity: (1, 50, 32)
[PASS] L14 | Loss=1.908389 | Grad=137.480851



CELL 6 — INTEGRATION RESULTS
L8 | Executed: 8 | Tokens: 229 | Expert KV: 16 | Velocity: (1, 50, 32) | Loss: 1.830594
L10 | Executed: 10 | Tokens: 229 | Expert KV: 16 | Velocity: (1, 50, 32) | Loss: 1.822181
L14 | Executed: 14 | Tokens: 229 | Expert KV: 16 | Velocity: (1, 50, 32) | Loss: 1.908389

[PASS] L8 executed 8 layers
[PASS] L10 executed 10 layers
[PASS] L14 executed 14 layers
[PASS] Native Expert 16 layers unchanged
[PASS] 16 KV cache layers
[PASS] Native velocity shape 50 x 32
[PASS] Adapter backward
[PASS] No model weights updated
[PASS] Original full LFM model restored

CELL 6 COMPLETE


In [8]:
# ============================================================
# CELL 7 — REAL DATA + LFM NATIVE INTEGRATION
# ============================================================

import os, gc, json, time, hashlib, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from huggingface_hub import HfApi, hf_hub_download
from transformers import DynamicCache
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.smolvlm_with_expert import apply_rope
from PIL import Image

SEED = 42
STEPS = 10000
CHUNK = 50
DIM = 6
PADDED_DIM = 32
DEPTHS = (8, 10, 14)

DATASET_ID = "nauq-hna/clean_290_60annot_v3"
LFM_ID = "LiquidAI/LFM2.5-VL-450M"
OUT = Path("/kaggle/working/VLA_LFM25_NATIVE_seed42")
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert "lfm" in globals()
assert "lfm_lm" in globals()
assert "lfm_processor" in globals()
assert "lfm_adapter" in globals()
assert "policy" in globals()
assert len(lfm_lm.layers) == 16

m = policy.model
ve = m.vlm_with_expert
expert = ve.lm_expert

assert len(expert.layers) == 16
assert ve.attention_mode == "cross_attn"
assert m.config.chunk_size == 50

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

api = HfApi(token=HF_TOKEN)

DATA_REVISION = api.repo_info(
    DATASET_ID,
    repo_type="dataset"
).sha

LFM_REVISION = api.model_info(LFM_ID).sha

metadata_file = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename="meta/info.json",
    revision=DATA_REVISION,
    token=HF_TOKEN,
)

with open(metadata_file, encoding="utf-8") as f:
    info = json.load(f)

FPS = float(info["fps"])
assert int(info["total_episodes"]) == 290

print("Dataset:", DATA_REVISION)
print("LFM:", LFM_REVISION)
print("FPS:", FPS)

# ------------------------------------------------------------
# SPLIT: 100 TRAIN / 30 VAL / 60 TEST / 100 UNUSED
# ------------------------------------------------------------

permutation = np.random.default_rng(SEED).permutation(290)

TRAIN_EPS = sorted(map(int, permutation[:100]))
VAL_EPS = sorted(map(int, permutation[100:130]))
TEST_EPS = sorted(map(int, permutation[130:190]))
UNUSED_EPS = sorted(map(int, permutation[190:]))

assert len(set(TRAIN_EPS + VAL_EPS + TEST_EPS)) == 190

split_records = []

for split_name, eps in [
    ("train", TRAIN_EPS),
    ("val", VAL_EPS),
    ("test", TEST_EPS),
    ("unused", UNUSED_EPS),
]:
    for ep in eps:
        split_records.append({
            "episode": ep,
            "split": split_name,
        })

split_path = OUT / "split_seed42.csv"

pd.DataFrame(split_records).sort_values(
    "episode"
).to_csv(split_path, index=False)

SPLIT_HASH = hashlib.sha256(
    split_path.read_bytes()
).hexdigest()

print("[PASS] Split 100 / 30 / 60 / 100")

# ------------------------------------------------------------
# LOAD TRAIN + VALIDATION DATA
# ------------------------------------------------------------

deltas = {
    "action": [
        round(i / FPS, 6)
        for i in range(CHUNK)
    ]
}

ds_trainval = LeRobotDataset(
    DATASET_ID,
    episodes=TRAIN_EPS + VAL_EPS,
    delta_timestamps=deltas,
    revision=DATA_REVISION,
    token=HF_TOKEN,
    download_videos=True,
)

rows = ds_trainval.select_columns([
    "episode_index",
    "frame_index",
    "action",
    "observation.state",
]).to_pandas().reset_index(drop=True)

train_rows = rows[
    rows.episode_index.isin(TRAIN_EPS)
]

a = np.stack(
    train_rows["action"].to_numpy()
).astype(np.float32)

s = np.stack(
    train_rows["observation.state"].to_numpy()
).astype(np.float32)

assert a.shape[1] == 6
assert s.shape[1] == 6

# Training-only normalization
AM_np = a.mean(axis=0)
AS_np = np.maximum(a.std(axis=0), 1e-4)
SM_np = s.mean(axis=0)
SS_np = np.maximum(s.std(axis=0), 1e-4)

np.savez(
    OUT / "normalization_seed42.npz",
    AM=AM_np,
    AS=AS_np,
    SM=SM_np,
    SS=SS_np,
)

AM, AS, SM, SS = [
    torch.as_tensor(
        x, device=DEVICE, dtype=torch.float32
    )
    for x in (AM_np, AS_np, SM_np, SS_np)
]

print("[PASS] Training-only normalization")

# ------------------------------------------------------------
# VALID FULL ACTION WINDOWS
# ------------------------------------------------------------

valid = {}

for ep in TRAIN_EPS + VAL_EPS:

    group = rows[rows.episode_index == ep]

    final_frame = int(group.frame_index.max())

    candidates = group[
        group.frame_index <= final_frame - CHUNK + 1
    ]

    valid[ep] = candidates.index.to_numpy(dtype=int)

    assert len(valid[ep]) > 0, (
        f"No complete action window in episode {ep}"
    )

rng = np.random.default_rng(SEED)

TRAIN_PLAN = []

for step in range(1, STEPS + 1):

    ep = TRAIN_EPS[(step - 1) % len(TRAIN_EPS)]

    TRAIN_PLAN.append({
        "step": step,
        "episode": ep,
        "index": int(rng.choice(valid[ep])),
    })

VAL_INDICES = [
    int(valid[ep][len(valid[ep]) // 2])
    for ep in VAL_EPS
]

pd.DataFrame(TRAIN_PLAN).to_csv(
    OUT / "training_plan.csv",
    index=False,
)

print("[PASS] 10000 training samples scheduled")

# ------------------------------------------------------------
# READ ACTUAL CAMERA / STATE / ACTION SAMPLE
# ------------------------------------------------------------

CAM_KEYS = [
    "observation.images.cam_head",
    "observation.images.cam_side",
    "observation.images.cam_wrist",
]

INSTRUCTION = (
    "pick up the orange cube and place it into the bowl"
)

def to_pil_lfm(x):

    x = torch.as_tensor(x).detach().cpu().float()

    if x.ndim == 4:
        x = x[0]

    assert x.ndim == 3

    if x.shape[-1] in (3, 4) and x.shape[0] != 3:
        x = x.permute(2, 0, 1)

    x = x[:3]

    if x.max().item() > 1.5:
        x = x / 255.0

    arr = (
        x.clamp(0, 1)
        .permute(1, 2, 0)
        .contiguous()
        .numpy()
    )

    return Image.fromarray(
        (arr * 255).round().astype(np.uint8)
    )


def read_sample(index, dataset=None):

    if dataset is None:
        dataset = ds_trainval

    raw = dataset[int(index)]

    if "action_is_pad" in raw:
        assert not bool(
            torch.as_tensor(raw["action_is_pad"]).any()
        ), f"Padded actions at index {index}"

    images = [
        to_pil_lfm(raw[k])
        for k in CAM_KEYS
    ]

    state = torch.as_tensor(
        raw["observation.state"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, DIM)

    actions = torch.as_tensor(
        raw["action"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, CHUNK, DIM)

    return (
        images,
        (state - SM) / SS,
        (actions - AM) / AS,
        actions,
    )

# ------------------------------------------------------------
# TRUE TRUNCATED LFM BACKBONE FEATURES
# ------------------------------------------------------------

ALL_LFM_LAYERS = list(lfm_lm.layers)

lfm.eval().requires_grad_(False)

# Original SmolVLM is not required when the Expert reads
# externally constructed KV caches. Release its GPU memory.
ve.vlm.eval().requires_grad_(False).cpu()
torch.cuda.empty_cache()

ve.num_vlm_layers = 16
m.config.use_cache = True

def set_lfm_depth(depth):

    assert depth in DEPTHS

    lfm_lm.layers = nn.ModuleList(
        ALL_LFM_LAYERS[:depth]
    )

    lfm_lm.config.num_hidden_layers = depth
    lfm.config.text_config.num_hidden_layers = depth

    assert len(lfm_lm.layers) == depth


def prepare_lfm_inputs(images):

    message = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Head camera:"},
            {"type": "image", "image": images[0]},
            {"type": "text", "text": "Side camera:"},
            {"type": "image", "image": images[1]},
            {"type": "text", "text": "Wrist camera:"},
            {"type": "image", "image": images[2]},
            {"type": "text", "text": INSTRUCTION},
        ],
    }]

    data = lfm_processor.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    result = {}

    for k, v in data.items():

        if not isinstance(v, torch.Tensor):
            result[k] = v
            continue

        if k == "pixel_values":
            result[k] = v.to(
                device=DEVICE,
                dtype=next(lfm.parameters()).dtype,
            )
        else:
            result[k] = v.to(DEVICE)

    return result


@torch.no_grad()
def extract_lfm_features(images, depth):

    assert len(lfm_lm.layers) == depth

    data = prepare_lfm_inputs(images)

    # Run multimodal backbone without LM vocabulary head.
    output = lfm.model(
        **data,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

    hs = output.hidden_states

    assert hs is not None, "No hidden states returned"
    assert len(hs) == depth + 1, (
        f"L{depth}: expected {depth+1} hidden states, "
        f"received {len(hs)}"
    )

    features = [
        hs[i + 1].detach()
        for i in range(depth)
    ]

    assert all(
        x.ndim == 3 and x.shape[-1] == 1024
        for x in features
    )

    return features

# ------------------------------------------------------------
# ORIGINAL STATE PROJECTION + COMPATIBILITY BRIDGE
# ------------------------------------------------------------

if "lfm_state_bridge" not in globals():
    lfm_state_bridge = nn.Linear(
        m.state_proj.out_features, 1024
    ).to(
        device=DEVICE,
        dtype=next(lfm_adapter.parameters()).dtype,
    )

lfm_adapter = lfm_adapter.to(DEVICE)
lfm_state_bridge = lfm_state_bridge.to(DEVICE)

def make_lfm_cache(features, depth, state):

    state32 = F.pad(state, (0, 26))

    native_state = m.state_proj(
        state32.to(m.state_proj.weight.dtype)
    )

    state_token = lfm_state_bridge(
        native_state.to(
            lfm_state_bridge.weight.dtype
        )
    ).unsqueeze(1)

    cache = DynamicCache()
    mapping = layer_mapping(depth)

    for i, src in enumerate(mapping):

        proj = lfm_adapter.projections[i]

        x = torch.cat([
            features[src].to(state_token.dtype),
            state_token,
        ], dim=1)

        x = x.to(proj["key"].weight.dtype)

        B, P, _ = x.shape

        k = proj["key"](x).reshape(B, P, 5, 64)
        v = proj["value"](x).reshape(B, P, 5, 64)

        positions = torch.arange(
            P, device=DEVICE
        ).unsqueeze(0).expand(B, -1)

        # RoPE for native-compatible stored prefix keys
        k = apply_rope(k, positions)

        cache.update(
            k.transpose(1, 2).contiguous(),
            v.transpose(1, 2).contiguous(),
            i,
        )

    mask = torch.ones(
        state.shape[0],
        P,
        dtype=torch.bool,
        device=DEVICE,
    )

    return cache, mask

# ------------------------------------------------------------
# NATIVE SMOLVLA FLOW MATCHING
# ------------------------------------------------------------

def make_fm_noise(seed):

    gen = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    noise = torch.randn(
        (1, CHUNK, PADDED_DIM),
        generator=gen,
        device=DEVICE,
    )

    beta = np.random.default_rng(
        20000 + int(seed)
    ).beta(1.5, 1.0)

    tau = torch.tensor(
        [beta * 0.999 + 0.001],
        dtype=torch.float32,
        device=DEVICE,
    )

    return noise, tau


def flow_loss(features, depth, state, target6, seed):

    target = F.pad(target6, (0, 26))

    noise, tau = make_fm_noise(seed)

    xt = (
        tau[:, None, None] * noise
        + (1 - tau[:, None, None]) * target
    )

    target_velocity = noise - target

    cache, mask = make_lfm_cache(
        features, depth, state
    )

    predicted_velocity = m.denoise_step(
        prefix_pad_masks=mask,
        past_key_values=cache,
        x_t=xt.to(m.action_in_proj.weight.dtype),
        timestep=tau,
    )

    assert predicted_velocity.shape == (1, 50, 32)

    # Loss on the six real action channels.
    return (
        predicted_velocity[:, :, :6].float()
        - target_velocity[:, :, :6]
    ).square().mean()


@torch.no_grad()
def predict_lfm(features, depth, state, seed):

    cache, mask = make_lfm_cache(
        features, depth, state
    )

    gen = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    x = torch.randn(
        (1, CHUNK, PADDED_DIM),
        device=DEVICE,
        generator=gen,
    )

    for n in range(10):

        tau = torch.full(
            (1,),
            1.0 - n / 10,
            device=DEVICE,
        )

        velocity = m.denoise_step(
            prefix_pad_masks=mask,
            past_key_values=cache,
            x_t=x.to(m.action_in_proj.weight.dtype),
            timestep=tau,
        )

        x = x - velocity.float() / 10

    return x[:, :, :6] * AS + AM

# ------------------------------------------------------------
# REAL DATA FORWARD/BACKWARD PRECHECK
# ------------------------------------------------------------

print("\n=== REAL DATA PRECHECK ===")

probe = TRAIN_PLAN[0]
images, state, target, gt = read_sample(
    probe["index"]
)

set_lfm_depth(8)

features = extract_lfm_features(images, 8)

lfm_adapter.zero_grad(set_to_none=True)
lfm_state_bridge.zero_grad(set_to_none=True)
m.zero_grad(set_to_none=True)

loss = flow_loss(
    features, 8, state, target, seed=123
)

assert torch.isfinite(loss)

loss.backward()

grads = [
    p.grad
    for p in lfm_adapter.parameters()
    if p.grad is not None
]

assert grads
assert all(torch.isfinite(g).all() for g in grads)
assert any(g.float().abs().sum() > 0 for g in grads)

print("[PASS] Real 3 cameras")
print("[PASS] Real state/action")
print("[PASS] Actual L8 forward")
print("[PASS] Native expert flow loss:", loss.item())
print("[PASS] Adapter gradients")

m.zero_grad(set_to_none=True)
lfm_adapter.zero_grad(set_to_none=True)
lfm_state_bridge.zero_grad(set_to_none=True)

print("\nCELL 7 COMPLETE")

info.json:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Dataset: 47b0decdb7bf0a80761f5581b09e0bffa7df6d4b
LFM: fc6221ca597f3315e4f82fc2df606783267b34ba
FPS: 30.0
[PASS] Split 100 / 30 / 60 / 100


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[PASS] Training-only normalization


[PASS] 10000 training samples scheduled



=== REAL DATA PRECHECK ===


[PASS] Real 3 cameras
[PASS] Real state/action
[PASS] Actual L8 forward
[PASS] Native expert flow loss: 3.37253737449646
[PASS] Adapter gradients

CELL 7 COMPLETE


In [9]:
# =====================================================
# RESUME PATCH — PRESERVE L8, FIX L10/L14
# Place AFTER Cell 7 and BEFORE Cell 8
# =====================================================

from pathlib import Path
import shutil
import hashlib
import pandas as pd
import torch

OUT = Path("/kaggle/working/VLA_LFM25_NATIVE_seed42")
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------
# 1. RESTORE PREVIOUS L8 OUTPUT
# -----------------------------------------------------

if not (OUT / "L8_latest.pt").exists():

    candidates = list(
        Path("/kaggle/input").rglob("L8_latest.pt")
    )

    assert len(candidates) == 1, (
        "Cannot find previous L8 output. "
        "Attach the old notebook output using Add Input. "
        f"Found: {candidates}"
    )

    source = candidates[0].parent

    files_to_restore = [
        "L8_best.pt",
        "L8_latest.pt",
        "training_steps.csv",
        "validation_history.csv",
        "training_summary.csv",
        "normalization_seed42.npz",
        "split_seed42.csv",
        "training_plan.csv",
    ]

    # Verify the experiment uses exactly the same split.
    old_split_hash = hashlib.sha256(
        (source / "split_seed42.csv").read_bytes()
    ).hexdigest()

    assert old_split_hash == SPLIT_HASH, (
        "Split mismatch! Do not reuse this checkpoint."
    )

    for filename in files_to_restore:
        src = source / filename
        assert src.exists(), f"Missing {src}"

        shutil.copy2(src, OUT / filename)

        print("[RESTORED]", filename)

else:
    print("[INFO] Existing L8 checkpoint retained")

# -----------------------------------------------------
# 2. VERIFY L8 COMPLETION
# -----------------------------------------------------

logs = pd.read_csv(OUT / "training_steps.csv")

l8 = logs[logs["model"] == "L8"]

assert l8["step"].max() == 10000, (
    "L8 has not reached step 10000"
)

assert (OUT / "L8_best.pt").exists()
assert (OUT / "L8_latest.pt").exists()

print("\n[PASS] L8 completed 10000 steps")
print("[PASS] L8 best/latest checkpoints preserved")

# -----------------------------------------------------
# 3. FIX LFM FEATURE EXTRACTION
# -----------------------------------------------------
# Do not use output.hidden_states.
# Capture each actual executed layer directly.
# -----------------------------------------------------

@torch.no_grad()
def extract_lfm_features(images, depth):

    assert len(lfm_lm.layers) == depth, (
        f"Expected {depth} active layers, "
        f"found {len(lfm_lm.layers)}"
    )

    captured = [None] * depth
    hooks = []

    def make_hook(index):

        def hook(module, args, output):

            hidden = (
                output[0]
                if isinstance(output, (tuple, list))
                else output
            )

            captured[index] = hidden.detach()

        return hook

    try:

        for i in range(depth):

            hooks.append(
                lfm_lm.layers[i].register_forward_hook(
                    make_hook(i)
                )
            )

        data = prepare_lfm_inputs(images)

        _ = lfm.model(
            **data,
            output_hidden_states=False,
            use_cache=False,
            return_dict=True,
        )

    finally:

        for hook in hooks:
            hook.remove()

    assert all(x is not None for x in captured), (
        f"L{depth}: captured "
        f"{sum(x is not None for x in captured)}/{depth}"
    )

    assert all(
        x.ndim == 3
        and x.shape[-1] == 1024
        and torch.isfinite(x).all()
        for x in captured
    )

    return captured

print("\n[PASS] Dynamic LFM extractor patched")

# -----------------------------------------------------
# 4. PREFLIGHT L10 AND L14
# -----------------------------------------------------

probe = TRAIN_PLAN[0]

images, state, target, gt = read_sample(
    probe["index"]
)

for depth in (10, 14):

    set_lfm_depth(depth)

    features = extract_lfm_features(
        images, depth
    )

    assert len(features) == depth

    print(
        f"[PASS] L{depth}: "
        f"{len(features)} actual hidden states"
    )

# Restore depth for Cell 8's training loop.
set_lfm_depth(8)

print("\n" + "=" * 60)
print("RESUME PATCH COMPLETE")
print("L8 preserved — L10/L14 ready for training")
print("=" * 60)

[RESTORED] L8_best.pt


[RESTORED] L8_latest.pt
[RESTORED] training_steps.csv
[RESTORED] validation_history.csv
[RESTORED] training_summary.csv
[RESTORED] normalization_seed42.npz
[RESTORED] split_seed42.csv
[RESTORED] training_plan.csv

[PASS] L8 completed 10000 steps
[PASS] L8 best/latest checkpoints preserved

[PASS] Dynamic LFM extractor patched


[PASS] L10: 10 actual hidden states


[PASS] L14: 14 actual hidden states

RESUME PATCH COMPLETE
L8 preserved — L10/L14 ready for training


In [10]:
# ============================================================
# CELL 8 — LFM2.5 L8 / L10 / L14 NATIVE TRAINING
# ============================================================

import copy
import gc
import time
import torch
import pandas as pd
import numpy as np

STEPS = 10000
VAL_EVERY = 1000
LOG_EVERY = 100
LR = 2e-4

CORE_NAMES = [
    "state_proj",
    "action_in_proj",
    "action_out_proj",
    "action_time_mlp_in",
    "action_time_mlp_out",
]

# ------------------------------------------------------------
# INITIAL WEIGHTS — PRESERVE ACROSS ALL THREE RUNS
# ------------------------------------------------------------

def clone_cpu_state(module):
    return {
        k: v.detach().cpu().clone()
        for k, v in module.state_dict().items()
    }


def capture_native_weights():

    result = {
        name: clone_cpu_state(getattr(m, name))
        for name in CORE_NAMES
    }

    result["lm_expert"] = clone_cpu_state(
        ve.lm_expert
    )

    result["adapter"] = clone_cpu_state(
        lfm_adapter
    )

    result["state_bridge"] = clone_cpu_state(
        lfm_state_bridge
    )

    return result


if "BASE_NATIVE_LFM" not in globals():
    BASE_NATIVE_LFM = capture_native_weights()


def restore_weights(weights):

    for name in CORE_NAMES:
        getattr(m, name).load_state_dict(
            weights[name], strict=True
        )

    ve.lm_expert.load_state_dict(
        weights["lm_expert"], strict=True
    )

    lfm_adapter.load_state_dict(
        weights["adapter"], strict=True
    )

    lfm_state_bridge.load_state_dict(
        weights["state_bridge"], strict=True
    )


def trainable_modules():

    return [
        ve.lm_expert,
        *[getattr(m, name) for name in CORE_NAMES],
        lfm_adapter,
        lfm_state_bridge,
    ]


def trainable_parameters():

    return [
        p
        for module in trainable_modules()
        for p in module.parameters()
        if p.requires_grad
    ]


def reset_variant():

    restore_weights(BASE_NATIVE_LFM)

    for module in trainable_modules():
        module.requires_grad_(True)
        module.train()

    lfm.eval().requires_grad_(False)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

@torch.no_grad()
def validate_lfm(depth):

    for module in trainable_modules():
        module.eval()

    errors = []
    losses = []

    for idx in VAL_INDICES:

        images, state, target, ground_truth = (
            read_sample(idx)
        )

        features = extract_lfm_features(
            images, depth
        )

        fm = flow_loss(
            features, depth, state, target,
            seed=900000 + int(idx)
        )

        pred = predict_lfm(
            features, depth, state,
            seed=700000 + int(idx)
        )

        mae = (
            pred - ground_truth
        ).abs().mean()

        errors.append(float(mae.item()))
        losses.append(float(fm.item()))

    return (
        float(np.mean(errors)),
        float(np.mean(losses)),
    )


# ------------------------------------------------------------
# CHECKPOINTING
# ------------------------------------------------------------

def save_checkpoint(
    depth,
    step,
    best_step,
    best_mae,
    val_fm,
    elapsed,
    optimizer,
    improved,
):

    model_name = f"L{depth}"

    common = {
        "model": model_name,
        "depth": depth,
        "step": step,
        "best_step": best_step,
        "best_val_mae": best_mae,
        "val_fm_loss": val_fm,

        "dataset": DATASET_ID,
        "dataset_revision": DATA_REVISION,
        "lfm_model": LFM_ID,
        "lfm_revision": LFM_REVISION,
        "split_hash": SPLIT_HASH,

        "train_wall_seconds": elapsed,
        "model_state": capture_native_weights(),

        "AM": AM.detach().cpu(),
        "AS": AS.detach().cpu(),
        "SM": SM.detach().cpu(),
        "SS": SS.detach().cpu(),

        "architecture": (
            "Frozen LFM2.5-VL truncated backbone "
            "+ layerwise KV adapter "
            "+ original 16-layer SmolVLA Action Expert"
        ),

        "flow_convention": "noise_at_one",
        "denoising_steps": 10,
        "chunk_size": 50,
        "action_dim": 6,
        "expert_layers": 16,
    }

    if improved:
        torch.save(
            common,
            OUT / f"{model_name}_best.pt",
        )

    latest = dict(common)
    latest["optimizer_state"] = optimizer.state_dict()

    torch.save(
        latest,
        OUT / f"{model_name}_latest.pt",
    )


# ------------------------------------------------------------
# LOAD EXISTING LOGS IF RERUNNING IN SAME KERNEL
# ------------------------------------------------------------

def read_existing_csv(filename):

    path = OUT / filename

    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path).to_dict("records")

    return []


train_logs = read_existing_csv("training_steps.csv")
val_logs = read_existing_csv("validation_history.csv")
summaries = read_existing_csv("training_summary.csv")


def save_logs():

    if train_logs:
        pd.DataFrame(train_logs).drop_duplicates(
            ["model", "step"], keep="last"
        ).to_csv(
            OUT / "training_steps.csv",
            index=False,
        )

    if val_logs:
        pd.DataFrame(val_logs).drop_duplicates(
            ["model", "step"], keep="last"
        ).to_csv(
            OUT / "validation_history.csv",
            index=False,
        )

    if summaries:
        pd.DataFrame(summaries).drop_duplicates(
            ["model"], keep="last"
        ).to_csv(
            OUT / "training_summary.csv",
            index=False,
        )


# ------------------------------------------------------------
# TRAIN ALL THREE MODELS SEQUENTIALLY
# ------------------------------------------------------------

for depth in DEPTHS:

    model_name = f"L{depth}"

    print("\n" + "=" * 70)
    print(f"TRAINING {model_name} — 10000 STEPS")
    print("=" * 70, flush=True)

    set_lfm_depth(depth)
    reset_variant()

    optimizer = torch.optim.AdamW(
        trainable_parameters(),
        lr=LR,
        weight_decay=0.01,
    )

    start_step = 1
    best_step = 0
    best_mae = float("inf")
    previous_elapsed = 0.0

    latest_file = OUT / f"{model_name}_latest.pt"

    if latest_file.exists():

        ck = torch.load(
            latest_file,
            map_location="cpu",
            weights_only=False,
        )

        assert ck["depth"] == depth
        assert ck["dataset_revision"] == DATA_REVISION
        assert ck["lfm_revision"] == LFM_REVISION
        assert ck["split_hash"] == SPLIT_HASH

        restore_weights(ck["model_state"])
        optimizer.load_state_dict(
            ck["optimizer_state"]
        )

        start_step = int(ck["step"]) + 1
        best_step = int(ck["best_step"])
        best_mae = float(ck["best_val_mae"])
        previous_elapsed = float(
            ck["train_wall_seconds"]
        )

        print(
            f"[RESUME] {model_name} "
            f"from step {start_step}",
            flush=True,
        )

    if start_step > STEPS:
        print(
            f"[SKIP] {model_name} already trained",
            flush=True,
        )
        del optimizer
        continue

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    print(
        "Expert layers:", len(ve.lm_expert.layers)
    )
    print(
        "Executed backbone layers:", len(lfm_lm.layers)
    )
    print(
        "Trainable parameters:",
        f"{sum(p.numel() for p in trainable_parameters()):,}"
    )

    for step in range(start_step, STEPS + 1):

        item = TRAIN_PLAN[step - 1]

        step_start = time.perf_counter()

        images, state, target, _ = read_sample(
            item["index"]
        )

        # Pretrained backbone is frozen.
        features = extract_lfm_features(
            images, depth
        )

        for module in trainable_modules():
            module.train()

        optimizer.zero_grad(set_to_none=True)

        # Original Native Expert with native 6-channel loss.
        loss = flow_loss(
            features,
            depth,
            state,
            target,
            seed=step,
        )

        assert torch.isfinite(loss), (
            f"{model_name} nonfinite loss at step {step}"
        )

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters(),
            max_norm=1.0,
            error_if_nonfinite=True,
        )

        optimizer.step()

        torch.cuda.synchronize()

        train_logs.append({
            "model": model_name,
            "step": step,
            "episode": item["episode"],
            "train_fm_loss": float(loss.item()),
            "grad_norm": float(grad_norm.item()),
            "step_seconds": (
                time.perf_counter() - step_start
            ),
        })

        if step == start_step or step % LOG_EVERY == 0:

            print(
                f"[{model_name}] "
                f"{step}/{STEPS} "
                f"loss={loss.item():.6f} "
                f"grad={grad_norm.item():.4f}",
                flush=True,
            )

            save_logs()

        # ----------------------------------------------------
        # VALIDATION AND BEST / LATEST CHECKPOINTS
        # ----------------------------------------------------

        if step % VAL_EVERY == 0:

            val_mae, val_fm = validate_lfm(depth)

            improved = val_mae < best_mae

            if improved:
                best_mae = val_mae
                best_step = step

            elapsed = (
                previous_elapsed
                + time.perf_counter() - start_time
            )

            val_logs.append({
                "model": model_name,
                "step": step,
                "val_mae": val_mae,
                "val_fm_loss": val_fm,
                "best_val_mae": best_mae,
                "best_step": best_step,
                "selected": improved,
            })

            save_checkpoint(
                depth=depth,
                step=step,
                best_step=best_step,
                best_mae=best_mae,
                val_fm=val_fm,
                elapsed=elapsed,
                optimizer=optimizer,
                improved=improved,
            )

            save_logs()

            print(
                f"[{model_name} VALID] "
                f"step={step} "
                f"MAE={val_mae:.6f} "
                f"FM={val_fm:.6f} "
                f"best={best_step}",
                flush=True,
            )

    elapsed = (
        previous_elapsed
        + time.perf_counter() - start_time
    )

    summaries = [
        row for row in summaries
        if row["model"] != model_name
    ]

    summaries.append({
        "model": model_name,
        "steps": STEPS,
        "best_step": best_step,
        "best_val_mae": best_mae,
        "training_wall_seconds": elapsed,
        "trainable_parameters": sum(
            p.numel()
            for p in trainable_parameters()
        ),
        "peak_train_vram_gib": (
            torch.cuda.max_memory_allocated() / 2**30
        ),
    })

    save_logs()

    print(
        f"[PASS] {model_name} TRAINING COMPLETE",
        flush=True,
    )

    del optimizer
    gc.collect()
    torch.cuda.empty_cache()

# Restore full original LFM layers after the training.
lfm_lm.layers = nn.ModuleList(ALL_LFM_LAYERS)
lfm_lm.config.num_hidden_layers = 16
lfm.config.text_config.num_hidden_layers = 16

print("\n" + "=" * 70)
print("LFM TRAINING FINISHED")
print("=" * 70)

print(pd.DataFrame(summaries).to_string(index=False))
print("Output:", OUT)

for depth in DEPTHS:
    assert (OUT / f"L{depth}_best.pt").exists()
    assert (OUT / f"L{depth}_latest.pt").exists()

print("[PASS] L8 / L10 / L14 best/latest checkpoints")
print("[PASS] Combined training CSVs")


TRAINING L8 — 10000 STEPS


[RESUME] L8 from step 10001


[SKIP] L8 already trained



TRAINING L10 — 10000 STEPS


Expert layers: 16
Executed backbone layers: 10
Trainable parameters: 111,361,056


...

[L14 VALID] step=5000 MAE=21.558954 FM=1.207721 best=5000


[L14] 5100/10000 loss=1.168794 grad=19.0302


[L14] 5200/10000 loss=0.959162 grad=16.8032


[L14] 5300/10000 loss=0.755443 grad=14.0297


[L14] 5400/10000 loss=1.101156 grad=15.2258


In [ ]:
# ============================================================
# CELL 9 — LFM2.5 L8 / L10 / L14 FINAL BENCHMARK
# ============================================================
# Uses BEST checkpoints from Cell 8.
#
# Protocol:
#   60 held-out test episodes
#   5 windows per episode
#   3 noise seeds per window
#   10 Euler denoising steps
#   50 actions x 6 real dimensions
#
# 300 windows/model
# 900 predictions/model
# 2700 predictions total
#
# No training, no optimizer updates.
# ============================================================

import gc
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# ============================================================
# 1. CONFIGURATION
# ============================================================

OUT = Path("/kaggle/working/VLA_LFM25_NATIVE_seed42")
OUT.mkdir(parents=True, exist_ok=True)

DEPTHS = (8, 10, 14)
TEST_SEEDS = (0, 1, 2)

WINDOWS_PER_EP = 5
DENOISING_STEPS = 10

CHUNK = 50
DIM = 6

assert len(TRAIN_EPS) == 100
assert len(VAL_EPS) == 30
assert len(TEST_EPS) == 60

assert len(ALL_LFM_LAYERS) == 16
assert len(ve.lm_expert.layers) == 16

# Training must finish before testing.
for depth in DEPTHS:
    assert (
        OUT / f"L{depth}_best.pt"
    ).exists(), f"Missing L{depth} best checkpoint"

for filename in [
    "training_steps.csv",
    "validation_history.csv",
    "training_summary.csv",
    "split_seed42.csv",
    "training_plan.csv",
    "normalization_seed42.npz",
]:
    assert (OUT / filename).exists(), (
        f"Missing training artifact: {filename}"
    )

print("=" * 70)
print("LFM2.5 NATIVE SMOLVLA FINAL BENCHMARK")
print("=" * 70)
print("Models: L8, L10, L14")
print("Test episodes:", len(TEST_EPS))
print("Windows/model:", 60 * WINDOWS_PER_EP)
print("Predictions/model:", 60 * WINDOWS_PER_EP * 3)
print("Total predictions:", 2700)
print("Output:", OUT)

# ============================================================
# 2. VERIFY TRAINING RESULTS
# ============================================================

training_df = pd.read_csv(
    OUT / "training_steps.csv"
)

validation_df = pd.read_csv(
    OUT / "validation_history.csv"
)

training_summary_df = pd.read_csv(
    OUT / "training_summary.csv"
)

training_df = training_df.drop_duplicates(
    ["model", "step"],
    keep="last",
)

validation_df = validation_df.drop_duplicates(
    ["model", "step"],
    keep="last",
)

assert set(training_summary_df.model) == {
    "L8", "L10", "L14"
}

for depth in DEPTHS:
    name = f"L{depth}"

    assert len(
        training_df[training_df.model == name]
    ) == 10000, f"{name} training incomplete"

    assert len(
        validation_df[validation_df.model == name]
    ) == 10, f"{name} validation incomplete"

print("[PASS] 30000 training steps")
print("[PASS] 30 validation records")
print("[PASS] Three training summaries")

# ============================================================
# 3. LOAD HELD-OUT TEST DATA
# ============================================================

print("\nLoading test dataset...")

test_ds = LeRobotDataset(
    DATASET_ID,
    episodes=TEST_EPS,
    delta_timestamps=deltas,
    revision=DATA_REVISION,
    token=HF_TOKEN,
    download_videos=True,
)

frame_table = test_ds.select_columns([
    "episode_index",
    "frame_index",
]).to_pandas().reset_index(drop=True)

assert set(
    frame_table.episode_index.unique()
) == set(TEST_EPS)

print("[PASS] 60 held-out test episodes loaded")

# ============================================================
# 4. DETERMINISTIC TEST PLAN
# ============================================================

test_plan = []

for ep in sorted(TEST_EPS):

    group = frame_table[
        frame_table.episode_index == ep
    ]

    last_frame = int(group.frame_index.max())

    candidates = group[
        group.frame_index <= last_frame - CHUNK + 1
    ]

    assert len(candidates) >= WINDOWS_PER_EP, ep

    positions = np.linspace(
        0,
        len(candidates) - 1,
        WINDOWS_PER_EP + 2,
    )[1:-1].round().astype(int)

    assert len(set(positions)) == WINDOWS_PER_EP

    for pos in positions:

        index = int(candidates.index[int(pos)])
        row = candidates.iloc[int(pos)]

        test_plan.append({
            "episode": int(ep),
            "frame_index": int(row.frame_index),
            "index": index,
        })

assert len(test_plan) == 300

plan_df = pd.DataFrame(test_plan)

plan_file = OUT / "test_plan.csv"

if plan_file.exists():
    previous_plan = pd.read_csv(plan_file)
    assert previous_plan.equals(plan_df), (
        "Existing test plan differs from current plan"
    )

plan_df.to_csv(plan_file, index=False)

TEST_PLAN_HASH = hashlib.sha256(
    plan_file.read_bytes()
).hexdigest()

print("[PASS] Identical deterministic 300-window plan")

# ============================================================
# 5. METRICS
# ============================================================

def gt_hash(gt):

    array = (
        gt.detach()
        .cpu()
        .float()
        .contiguous()
        .numpy()
    )

    return hashlib.sha256(
        array.tobytes()
    ).hexdigest()


def calculate_metrics(pred, gt):

    assert pred.shape == gt.shape == (
        1, 50, 6
    )

    pred = pred.float()
    gt = gt.float()

    error = pred - gt

    dp = pred[:, 1:] - pred[:, :-1]
    dg = gt[:, 1:] - gt[:, :-1]

    ddp = dp[:, 1:] - dp[:, :-1]
    ddg = dg[:, 1:] - dg[:, :-1]

    result = {
        "mae": error.abs().mean().item(),

        "mse": error.square().mean().item(),

        "normalized_mae": (
            error / AS.reshape(1, 1, 6)
        ).abs().mean().item(),

        "delta_error": (
            dp - dg
        ).abs().mean().item(),

        "second_diff_error": (
            ddp - ddg
        ).abs().mean().item(),

        "pred_smoothness": (
            dp.abs().mean().item()
        ),

        "demo_smoothness": (
            dg.abs().mean().item()
        ),
    }

    for joint in range(6):

        result[f"joint_{joint}_mae"] = (
            error[:, :, joint].abs().mean().item()
        )

    return result


def make_test_seed(episode, frame_index, seed_id):

    # Same seed convention as the SmolVLA A benchmark.
    return (
        50000
        + int(episode) * 1000000
        + int(frame_index)
        + int(seed_id) * 100000
    )


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


# ============================================================
# 6. CHECKPOINT RESTORATION
# ============================================================

def restore_best_lfm(depth):

    name = f"L{depth}"

    checkpoint = torch.load(
        OUT / f"{name}_best.pt",
        map_location="cpu",
        weights_only=False,
    )

    assert checkpoint["model"] == name
    assert checkpoint["depth"] == depth

    assert (
        checkpoint["dataset_revision"]
        == DATA_REVISION
    )

    assert (
        checkpoint["lfm_revision"]
        == LFM_REVISION
    )

    assert (
        checkpoint["split_hash"]
        == SPLIT_HASH
    )

    # Original Native SmolVLA Expert and projections.
    restore_weights(
        checkpoint["model_state"]
    )

    # Truly execute the selected LFM depth.
    set_lfm_depth(depth)

    assert len(lfm_lm.layers) == depth
    assert len(ve.lm_expert.layers) == 16

    lfm.eval()
    m.eval()
    lfm_adapter.eval()
    lfm_state_bridge.eval()

    m.config.use_cache = True

    return checkpoint


# ============================================================
# 7. PARAMETER COUNT
# ============================================================

def count_variant_parameters():

    modules = [
        ve.lm_expert,

        m.state_proj,
        m.action_in_proj,
        m.action_out_proj,
        m.action_time_mlp_in,
        m.action_time_mlp_out,

        lfm_adapter,
        lfm_state_bridge,
    ]

    # Only the active LFM multimodal backbone.
    backbone_params = sum(
        p.numel()
        for p in lfm.model.parameters()
    )

    expert_adapter_params = sum(
        p.numel()
        for module in modules
        for p in module.parameters()
    )

    return {
        "total": backbone_params + expert_adapter_params,
        "trainable": expert_adapter_params,
        "expert": sum(
            p.numel()
            for p in ve.lm_expert.parameters()
        ),
        "adapter": sum(
            p.numel()
            for p in lfm_adapter.parameters()
        ),
    }


# ============================================================
# 8. RESUMABLE TEST OUTPUTS
# ============================================================

details_file = OUT / "test_details.csv"
trajectories_file = OUT / "action_trajectories.csv"

if details_file.exists() and details_file.stat().st_size:

    all_details = pd.read_csv(
        details_file
    ).to_dict("records")

    print(
        "[RESUME] Existing test rows:",
        len(all_details)
    )

else:
    all_details = []


if (
    trajectories_file.exists()
    and trajectories_file.stat().st_size
):

    all_trajectories = pd.read_csv(
        trajectories_file
    ).to_dict("records")

else:
    all_trajectories = []


def flush_test_progress():

    if all_details:

        pd.DataFrame(
            all_details
        ).drop_duplicates(
            [
                "model",
                "episode",
                "frame_index",
                "noise_seed",
            ],
            keep="last",
        ).to_csv(
            details_file,
            index=False,
        )

    if all_trajectories:

        pd.DataFrame(
            all_trajectories
        ).drop_duplicates(
            [
                "model",
                "episode",
                "frame_index",
                "timestep",
                "joint",
            ],
            keep="last",
        ).to_csv(
            trajectories_file,
            index=False,
        )


# ============================================================
# 9. SELECT ACTION TRAJECTORIES
# ============================================================

TRAJECTORY_EPS = sorted(TEST_EPS)[:3]

TRAJECTORY_WINDOWS = {
    (
        row["episode"],
        row["frame_index"],
    )
    for row in test_plan
    if (
        row["episode"] in TRAJECTORY_EPS
        and row["index"] == next(
            x["index"]
            for x in test_plan
            if x["episode"] == row["episode"]
        )
    )
}

assert len(TRAJECTORY_WINDOWS) == 3


def save_trajectory(model, ep, frame, pred, gt):

    p = pred[0].detach().cpu().numpy()
    g = gt[0].detach().cpu().numpy()

    for timestep in range(50):

        for joint in range(6):

            all_trajectories.append({
                "model": model,
                "episode": ep,
                "frame_index": frame,
                "timestep": timestep,
                "joint": joint,
                "prediction": float(
                    p[timestep, joint]
                ),
                "ground_truth": float(
                    g[timestep, joint]
                ),
            })


# ============================================================
# 10. RUN ALL THREE MODELS
# ============================================================

all_summaries = []

for depth in DEPTHS:

    model_name = f"L{depth}"

    print("\n" + "=" * 70)
    print("TESTING", model_name)
    print("=" * 70, flush=True)

    checkpoint = restore_best_lfm(depth)

    params = count_variant_parameters()

    print("Backbone layers:", depth)
    print("Expert layers:", len(ve.lm_expert.layers))
    print("Best step:", checkpoint["best_step"])
    print("Best validation MAE:", checkpoint["best_val_mae"])

    print(
        "Total active parameters:",
        f"{params['total']:,}"
    )

    print(
        "Expert parameters:",
        f"{params['expert']:,}"
    )

    # --------------------------------------------------------
    # WARMUP
    # --------------------------------------------------------

    first = test_plan[0]

    images, state, target, gt = read_sample(
        first["index"],
        test_ds,
    )

    with torch.no_grad():

        for _ in range(2):

            features = extract_lfm_features(
                images, depth
            )

            _ = predict_lfm(
                features,
                depth,
                state,
                seed=42,
            )

    cuda_sync()
    torch.cuda.reset_peak_memory_stats()

    # --------------------------------------------------------
    # TEST 300 WINDOWS
    # --------------------------------------------------------

    for window_number, item in enumerate(
        test_plan,
        start=1,
    ):

        ep = item["episode"]
        frame = item["frame_index"]

        key = (model_name, ep, frame)

        existing_rows = [
            r for r in all_details
            if (
                r["model"] == model_name
                and int(r["episode"]) == ep
                and int(r["frame_index"]) == frame
            )
        ]

        trajectory_required = (
            ep, frame
        ) in TRAJECTORY_WINDOWS

        existing_trajectory_count = sum(
            r["model"] == model_name
            and int(r["episode"]) == ep
            and int(r["frame_index"]) == frame
            for r in all_trajectories
        )

        # Resume completed windows.
        if (
            len(existing_rows) == 3
            and not (
                trajectory_required
                and existing_trajectory_count != 300
            )
        ):
            continue

        # Remove incomplete previous results for this window.
        all_details = [
            r for r in all_details
            if not (
                r["model"] == model_name
                and int(r["episode"]) == ep
                and int(r["frame_index"]) == frame
            )
        ]

        all_trajectories = [
            r for r in all_trajectories
            if not (
                r["model"] == model_name
                and int(r["episode"]) == ep
                and int(r["frame_index"]) == frame
            )
        ]

        images, state, target, gt = read_sample(
            item["index"],
            test_ds,
        )

        results_this_window = []
        predictions = []

        # ----------------------------------------------------
        # THREE NOISE SEEDS
        # ----------------------------------------------------

        for noise_seed in TEST_SEEDS:

            seed = make_test_seed(
                ep,
                frame,
                noise_seed,
            )

            # Full pipeline latency:
            # images -> LFM -> adapter -> native expert
            cuda_sync()
            started = time.perf_counter()

            with torch.no_grad():

                features = extract_lfm_features(
                    images,
                    depth,
                )

                pred = predict_lfm(
                    features,
                    depth,
                    state,
                    seed,
                )

            cuda_sync()

            latency_ms = (
                time.perf_counter() - started
            ) * 1000

            assert pred.shape == (1, 50, 6)
            assert torch.isfinite(pred).all()

            # Separate Flow Matching evaluation.
            # Not included in measured inference latency.
            with torch.no_grad():

                fm = flow_loss(
                    features,
                    depth,
                    state,
                    target,
                    seed,
                )

            assert torch.isfinite(fm)

            row = {
                "model": model_name,
                "Model": model_name,

                "episode": ep,
                "frame_index": frame,

                "noise_seed": noise_seed,
                "seed_value": seed,

                "gt_hash": gt_hash(gt),

                "latency_ms": latency_ms,
                "inference_ms": latency_ms,

                "fm_loss": float(fm.item()),

                **calculate_metrics(pred, gt),
            }

            predictions.append(pred.detach().cpu())
            results_this_window.append(row)

        # ----------------------------------------------------
        # SEED VARIATION
        # ----------------------------------------------------

        stack = torch.cat(
            predictions,
            dim=0,
        )

        seed_std = stack.std(
            dim=0,
            unbiased=False,
        ).mean().item()

        for row in results_this_window:
            row["seed_std"] = seed_std

        all_details.extend(results_this_window)

        # ----------------------------------------------------
        # ACTION TRAJECTORY
        # ----------------------------------------------------

        if trajectory_required:

            save_trajectory(
                model_name,
                ep,
                frame,
                predictions[0],
                gt.cpu(),
            )

        # ----------------------------------------------------
        # PERIODIC SAVE
        # ----------------------------------------------------

        if (
            window_number % 20 == 0
            or window_number == len(test_plan)
        ):

            flush_test_progress()

            model_count = sum(
                r["model"] == model_name
                for r in all_details
            )

            print(
                f"[{model_name}] "
                f"{window_number}/300 windows "
                f"| {model_count}/900 predictions",
                flush=True,
            )

    # --------------------------------------------------------
    # MODEL VALIDATION
    # --------------------------------------------------------

    model_df = pd.DataFrame([
        r for r in all_details
        if r["model"] == model_name
    ])

    assert len(model_df) == 900, (
        f"{model_name}: expected 900 predictions, "
        f"got {len(model_df)}"
    )

    assert model_df.episode.nunique() == 60

    assert not model_df.duplicated([
        "episode",
        "frame_index",
        "noise_seed",
    ]).any()

    # --------------------------------------------------------
    # AGGREGATE METRICS
    # --------------------------------------------------------

    unique_windows = model_df.drop_duplicates([
        "episode",
        "frame_index",
    ])

    mean_latency = float(
        model_df.latency_ms.mean()
    )

    train_info = training_summary_df[
        training_summary_df.model == model_name
    ].iloc[0]

    train_rows = training_df[
        training_df.model == model_name
    ]

    summary = {
        "model": model_name,
        "Model": model_name,

        "backbone": LFM_ID,
        "backbone_layers": depth,
        "expert_layers": 16,

        "test_episodes": 60,
        "test_windows": 300,
        "predictions": 900,

        "test_mae": float(model_df.mae.mean()),
        "test_mse": float(model_df.mse.mean()),
        "test_rmse": float(
            np.sqrt(model_df.mse.mean())
        ),

        "test_normalized_mae": float(
            model_df.normalized_mae.mean()
        ),

        "test_delta_error": float(
            model_df.delta_error.mean()
        ),

        "test_second_diff_error": float(
            model_df.second_diff_error.mean()
        ),

        "test_pred_smoothness": float(
            model_df.pred_smoothness.mean()
        ),

        "test_demo_smoothness": float(
            model_df.demo_smoothness.mean()
        ),

        "test_smoothness_ratio": float(
            model_df.pred_smoothness.mean()
            / max(
                model_df.demo_smoothness.mean(),
                1e-8,
            )
        ),

        "test_seed_std": float(
            unique_windows.seed_std.mean()
        ),

        "test_fm_loss": float(
            model_df.fm_loss.mean()
        ),

        "inference_latency_mean_ms": mean_latency,

        "inference_latency_p50_ms": float(
            model_df.latency_ms.median()
        ),

        "inference_latency_p95_ms": float(
            model_df.latency_ms.quantile(0.95)
        ),

        "chunks_per_second": float(
            1000.0 / mean_latency
        ),

        "peak_inference_vram_gib": float(
            torch.cuda.max_memory_allocated()
            / 2**30
        ),

        "parameters_total": params["total"],
        "parameters_trainable": params["trainable"],

        "native_expert_parameters": params["expert"],
        "kv_adapter_parameters": params["adapter"],

        "best_step": int(checkpoint["best_step"]),
        "best_val_mae": float(
            checkpoint["best_val_mae"]
        ),

        "train_wall_seconds": float(
            train_info["training_wall_seconds"]
        ),

        "train_step_mean_seconds": float(
            train_rows.step_seconds.mean()
        ),
    }

    for joint in range(6):

        summary[f"joint_{joint}_mae"] = float(
            model_df[f"joint_{joint}_mae"].mean()
        )

    # Baseline-A-compatible metric columns.
    summary.update({
        "Test_Episodes": summary["test_episodes"],
        "Test_Windows": summary["test_windows"],
        "Predictions": summary["predictions"],

        "MAE": summary["test_mae"],
        "MSE": summary["test_mse"],
        "RMSE": summary["test_rmse"],

        "Normalized_MAE": summary["test_normalized_mae"],
        "Delta_Error": summary["test_delta_error"],
        "Second_Diff_Error": summary["test_second_diff_error"],

        "Smoothness_Ratio": summary["test_smoothness_ratio"],
        "Seed_STD": summary["test_seed_std"],

        "Latency_mean_ms": summary[
            "inference_latency_mean_ms"
        ],

        "Latency_P50_ms": summary[
            "inference_latency_p50_ms"
        ],

        "Latency_P95_ms": summary[
            "inference_latency_p95_ms"
        ],

        "Throughput_chunks_per_s": summary[
            "chunks_per_second"
        ],

        "Total_Params_M": params["total"] / 1e6,

        "Trainable_Params_M": (
            params["trainable"] / 1e6
        ),

        "Train_wall_s": summary[
            "train_wall_seconds"
        ],

        "Best_Step": summary["best_step"],
        "Best_Val_MAE": summary["best_val_mae"],
    })

    for joint in range(6):
        summary[f"Joint_{joint}_MAE"] = (
            summary[f"joint_{joint}_mae"]
        )

    all_summaries.append(summary)

    flush_test_progress()

    pd.DataFrame(
        all_summaries
    ).to_csv(
        OUT / "test_summary.csv",
        index=False,
    )

    print(
        f"\n[PASS] {model_name} TEST COMPLETE"
    )

    print(
        "MAE:",
        round(summary["test_mae"], 6)
    )

    print(
        "RMSE:",
        round(summary["test_rmse"], 6)
    )

    print(
        "Latency:",
        round(
            summary["inference_latency_mean_ms"],
            2,
        ),
        "ms",
    )

    gc.collect()
    torch.cuda.empty_cache()

# ============================================================
# 11. FINAL CROSS-MODEL VALIDATION
# ============================================================

details_df = pd.DataFrame(all_details)

details_df = details_df.drop_duplicates(
    [
        "model",
        "episode",
        "frame_index",
        "noise_seed",
    ],
    keep="last",
)

summary_df = pd.DataFrame(all_summaries)

trajectories_df = pd.DataFrame(
    all_trajectories
).drop_duplicates(
    [
        "model",
        "episode",
        "frame_index",
        "timestep",
        "joint",
    ],
    keep="last",
)

assert len(details_df) == 2700, (
    f"Expected 2700 predictions; got {len(details_df)}"
)

assert len(summary_df) == 3

assert len(trajectories_df) == 2700, (
    f"Expected 2700 trajectory rows; "
    f"got {len(trajectories_df)}"
)

for model_name in ("L8", "L10", "L14"):

    df = details_df[
        details_df.model == model_name
    ]

    assert len(df) == 900
    assert df.episode.nunique() == 60

    assert len(
        df.drop_duplicates([
            "episode",
            "frame_index",
        ])
    ) == 300

# Identical windows, seed IDs and GT across models.
keys = [
    "episode",
    "frame_index",
    "noise_seed",
    "gt_hash",
]

reference = (
    details_df[details_df.model == "L8"]
    .sort_values(keys)
    [keys]
    .reset_index(drop=True)
)

for name in ("L10", "L14"):

    comparison = (
        details_df[details_df.model == name]
        .sort_values(keys)
        [keys]
        .reset_index(drop=True)
    )

    assert reference.equals(comparison), (
        f"Test protocol mismatch: {name}"
    )

print("\n[PASS] Identical test windows")
print("[PASS] Identical ground-truth hashes")
print("[PASS] 2700 predictions")
print("[PASS] 2700 trajectory records")

# ============================================================
# 12. PER-EPISODE METRICS
# ============================================================

per_episode_df = details_df.groupby(
    ["model", "episode"],
    as_index=False,
).agg(
    windows=("frame_index", "nunique"),
    predictions=("mae", "size"),
    mae=("mae", "mean"),
    mse=("mse", "mean"),
    normalized_mae=("normalized_mae", "mean"),
    delta_error=("delta_error", "mean"),
    second_diff_error=("second_diff_error", "mean"),
    fm_loss=("fm_loss", "mean"),
    latency_ms=("latency_ms", "mean"),
)

per_episode_df["rmse"] = np.sqrt(
    per_episode_df["mse"]
)

assert len(per_episode_df) == 180

# ============================================================
# 13. JOINT METRICS
# ============================================================

joint_rows = []

for model_name in ("L8", "L10", "L14"):

    df = details_df[
        details_df.model == model_name
    ]

    for joint in range(6):

        joint_rows.append({
            "model": model_name,
            "joint": joint,
            "mae": float(
                df[f"joint_{joint}_mae"].mean()
            ),
        })

joint_df = pd.DataFrame(joint_rows)

assert len(joint_df) == 18

# ============================================================
# 14. SAVE ALL FINAL CSVS
# ============================================================

details_df.to_csv(
    OUT / "test_details.csv",
    index=False,
)

summary_df.to_csv(
    OUT / "test_summary.csv",
    index=False,
)

per_episode_df.to_csv(
    OUT / "test_per_episode.csv",
    index=False,
)

trajectories_df.to_csv(
    OUT / "action_trajectories.csv",
    index=False,
)

joint_df.to_csv(
    OUT / "joint_metrics.csv",
    index=False,
)

# Baseline-A compatible summary export.
summary_df.to_csv(
    OUT / "full_metrics.csv",
    index=False,
)

# ============================================================
# 15. EXPERIMENT MANIFEST
# ============================================================

manifest = {
    "experiment": "LFM2.5-VL Native SmolVLA Expert",

    "models": ["L8", "L10", "L14"],

    "backbone": LFM_ID,
    "backbone_revision": LFM_REVISION,

    "native_expert": "SmolVLA",
    "native_expert_layers": 16,

    "adapter": "Layerwise KV Adapter",
    "attention_mode": "cross_attn",

    "dataset": DATASET_ID,
    "dataset_revision": DATA_REVISION,

    "seed": SEED,
    "split_hash": SPLIT_HASH,
    "test_plan_sha256": TEST_PLAN_HASH,

    "train_episodes": TRAIN_EPS,
    "validation_episodes": VAL_EPS,
    "test_episodes": TEST_EPS,
    "unused_episodes": UNUSED_EPS,

    "train_steps_per_model": 10000,
    "validation_interval": 1000,

    "test_windows_per_episode": WINDOWS_PER_EP,
    "test_noise_seeds": list(TEST_SEEDS),

    "denoising_steps": DENOISING_STEPS,
    "chunk_size": CHUNK,
    "action_dimensions": DIM,

    "cameras": CAM_KEYS,
    "instruction": INSTRUCTION,

    "normalization": "training episodes only",

    "flow_matching": "noise_at_one",

    "test_checkpoints": {
        f"L{d}": f"L{d}_best.pt"
        for d in DEPTHS
    },

    "notes": (
        "The original SmolVLA 16-layer Action Expert "
        "and native denoising function are retained. "
        "The external KV Adapter is a compatibility "
        "interface, not the original SmolVLA VLM prefill. "
        "The LFM backbone is frozen during training. "
        "Offline action regression metrics do not "
        "measure physical robot task success."
    ),
}

with open(
    OUT / "experiment_manifest.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# 16. RESTORE FULL LFM
# ============================================================

lfm_lm.layers = nn.ModuleList(
    ALL_LFM_LAYERS
)

lfm_lm.config.num_hidden_layers = 16
lfm.config.text_config.num_hidden_layers = 16

# ============================================================
# 17. FINAL FILE VALIDATION
# ============================================================

required_files = [
    "split_seed42.csv",
    "training_plan.csv",
    "normalization_seed42.npz",

    "training_steps.csv",
    "validation_history.csv",
    "training_summary.csv",

    "L8_best.pt",
    "L8_latest.pt",

    "L10_best.pt",
    "L10_latest.pt",

    "L14_best.pt",
    "L14_latest.pt",

    "test_plan.csv",
    "test_details.csv",
    "test_summary.csv",
    "test_per_episode.csv",
    "action_trajectories.csv",

    "joint_metrics.csv",
    "full_metrics.csv",
    "experiment_manifest.json",
]

for filename in required_files:

    path = OUT / filename

    assert path.exists(), filename
    assert path.stat().st_size > 0, filename

print("\n" + "=" * 70)
print("FINAL BENCHMARK RESULTS")
print("=" * 70)

print(
    summary_df[
        [
            "model",
            "test_mae",
            "test_rmse",
            "test_normalized_mae",
            "test_delta_error",
            "test_second_diff_error",
            "test_smoothness_ratio",
            "test_fm_loss",
            "inference_latency_mean_ms",
            "peak_inference_vram_gib",
            "best_step",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("SAVED OUTPUT FILES")
print("=" * 70)

for filename in required_files:

    path = OUT / filename

    print(
        "[SAVED]",
        filename,
        f"({path.stat().st_size:,} bytes)"
    )

print("\n[PASS] 3 models trained")
print("[PASS] 30000 optimizer steps")
print("[PASS] 30 validation records")
print("[PASS] 2700 test predictions")
print("[PASS] 180 per-episode results")
print("[PASS] 2700 action trajectory records")
print("[PASS] 6 best/latest checkpoints")
print("[PASS] Experiment manifest")
print("[PASS] All required files saved")

print("\nLFM2.5 NATIVE BENCHMARK COMPLETE")
print("OUTPUT:", OUT)